In [2]:
from pathlib import Path
import sys

project_root = Path.cwd().parents[2]
sys.path.insert(0, str(project_root))

In [6]:
from src.medmate.rag.ingest import PDFParser
from config.settings import RAG_IMAGES_PATH

In [7]:
"""
Folder-based ingestion script for MedlinePlus lab glossary PDFs.

Loops through every PDF in `LAB_GLOSSARY_DIR`, derives metadata from the
filename (pattern: `<test_name>_test.pdf`, e.g. `ldl_cholesterol_test.pdf`),
auto-detects page count, and runs the same batched PDFParser logic as your
original medication ingestion script -- but per file, accumulating into a
single combined `all_documents` list for bulk Qdrant upsert.

Assumptions (change if wrong):
  - filename pattern is `<snake_case_test_name>_test.pdf`
  - fig/tbl/formula offsets reset to 0 for EACH new pdf (per-file numbering)
  - a failed pdf is skipped, loop continues to the next file
  - page count is auto-detected via pypdf
"""

from pathlib import Path
from pypdf import PdfReader

# from your_parser_module import PDFParser  # <-- keep your existing import
# from your_config import RAG_IMAGES_PATH    # <-- keep your existing import

BATCH_SIZE = 10
LAB_GLOSSARY_DIR = Path(r"C:\Users\adity\OneDrive\Documents\MedMate Clinic Copilot\data\rag_docs\lab_glossary")


def build_metadata_from_filename(pdf_path: Path) -> dict:
    """
    Derives lab-test metadata from filenames like `ldl_cholesterol_test.pdf`.

    ldl_cholesterol_test.pdf -> {
        "test_name": "Ldl Cholesterol",
        "test_slug": "ldl_cholesterol",
        "document_source": "MedlinePlus",
        "doc_type": "lab_glossary",
    }

    Adjust `test_name.title()` below if you want custom capitalization
    (e.g. "LDL Cholesterol" instead of "Ldl Cholesterol") -- easiest fix is
    a small override dict for acronyms like LDL/HDL/A1C/TSH.
    """
    stem = pdf_path.stem  # "ldl_cholesterol_test"
    slug = stem[:-5] if stem.endswith("_test") else stem  # strip trailing "_test"

    # Acronym overrides so "ldl" -> "LDL" instead of "Ldl"
    ACRONYMS = {"ldl", "hdl", "a1c", "tsh", "bun", "cbc", "psa", "crp", "inr"}
    words = slug.split("_")
    display_words = [w.upper() if w.lower() in ACRONYMS else w.capitalize() for w in words]
    test_name = " ".join(display_words)

    return {
        "test_name": test_name,
        "test_slug": slug,
        "document_source": "MedlinePlus",
        "document_type": "lab_glossary",
    }


def get_total_pages(pdf_path: Path) -> int:
    reader = PdfReader(str(pdf_path))
    return len(reader.pages)


def ingest_single_pdf(pdf_path: Path) -> tuple[list, list, list]:
    """
    Runs the batched parse loop (mirrors your original script) for ONE pdf.
    Returns (documents, image_paths, failed_ranges) for that file.
    """
    document_metadata = build_metadata_from_filename(pdf_path)
    total_pages = get_total_pages(pdf_path)

    print(f"\n--- {pdf_path.name} ---")
    print(f"  metadata: {document_metadata}")
    print(f"  total_pages: {total_pages}")

    documents = []
    image_paths = []
    failed_ranges = []

    fig_offset, tbl_offset, formula_offset = 0, 0, 0

    for batch_start in range(1, total_pages + 1, BATCH_SIZE):
        batch_end = min(batch_start + BATCH_SIZE - 1, total_pages)
        print(f"  Processing pages {batch_start}-{batch_end}...")

        parser = PDFParser(
            images_output_dir=RAG_IMAGES_PATH,
            process_images=True,
            process_formulas=True,
            page_range=(batch_start, batch_end),
            figure_idx_offset=fig_offset,
            table_idx_offset=tbl_offset,
            formula_idx_offset=formula_offset,
            document_metadata=document_metadata,
        )

        try:
            parsed_doc = parser.parse(str(pdf_path))
            documents.extend(parsed_doc.documents)
            image_paths.extend(parsed_doc.image_paths)

            fig_offset += sum(1 for d in parsed_doc.documents if d.metadata.get("element_type") == "image")
            tbl_offset += sum(1 for d in parsed_doc.documents if d.metadata.get("element_type") == "table")
            formula_offset += sum(1 for d in parsed_doc.documents if d.metadata.get("element_type") == "formula")

            print(f"    -> OK, {len(parsed_doc.documents)} docs "
                  f"(offsets: fig={fig_offset}, tbl={tbl_offset}, formula={formula_offset})")

        except Exception as e:
            print(f"    FAILED pages {batch_start}-{batch_end}: {type(e).__name__}: {e}")
            failed_ranges.append((pdf_path.name, batch_start, batch_end))

    return documents, image_paths, failed_ranges


def main():
    pdf_files = sorted(LAB_GLOSSARY_DIR.glob("*.pdf"))
    print(f"Found {len(pdf_files)} PDFs in {LAB_GLOSSARY_DIR}")

    all_documents = []
    all_image_paths = []
    all_failed_ranges = []
    failed_files = []

    for pdf_path in pdf_files:
        try:
            documents, image_paths, failed_ranges = ingest_single_pdf(pdf_path)
            all_documents.extend(documents)
            all_image_paths.extend(image_paths)
            all_failed_ranges.extend(failed_ranges)
        except Exception as e:
            # file-level failure (e.g. corrupted pdf, can't even open it)
            print(f"  SKIPPING FILE {pdf_path.name}: {type(e).__name__}: {e}")
            failed_files.append(pdf_path.name)
            continue

    print("\n=== SUMMARY ===")
    print("PDFs processed:", len(pdf_files) - len(failed_files))
    print("PDFs skipped (file-level failure):", failed_files)
    print("Total documents:", len(all_documents))
    print("Total images:", len(all_image_paths))
    print("Failed batch ranges:", all_failed_ranges)

    return all_documents, all_image_paths


all_documents, all_image_paths = main()
    # Next step (not included here): bulk upsert `all_documents` into Qdrant

Found 9 PDFs in C:\Users\adity\OneDrive\Documents\MedMate Clinic Copilot\data\rag_docs\lab_glossary

--- blood_glucose_test.pdf ---
  metadata: {'test_name': 'Blood Glucose', 'test_slug': 'blood_glucose', 'document_source': 'MedlinePlus', 'document_type': 'lab_glossary'}
  total_pages: 9
  Processing pages 1-9...
[PDFParser] Formula detection ENABLED (fast, layout-only — no VLM)
[PDFParser] Pipeline options: process_images=True, process_formulas=True, do_formula_enrichment=False, describe_images=False


[INFO] 2026-07-21 21:28:14,021 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-21 21:28:14,058 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\adity\OneDrive\Documents\MedMate Clinic Copilot\venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-07-21 21:28:14,060 [RapidOCR] main.py:63: Using C:\Users\adity\OneDrive\Documents\MedMate Clinic Copilot\venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-07-21 21:28:14,242 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-21 21:28:14,242 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\adity\OneDrive\Documents\MedMate Clinic Copilot\venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-21 21:28:14,242 [RapidOCR] main.py:63: Using C:\Users\adity\OneDrive\Documents\MedMate Clinic Copilot\venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-21 21:28:14


[PDFParser] File     : blood_glucose_test.pdf
[PDFParser] Pages    : (1, 9)
[PDFParser] Images   : enabled
[PDFParser] Formulas : enabled (no VLM)
[PDFParser] Types    : {'PictureItem': 2, 'TextItem': 59, 'SectionHeaderItem': 13, 'ListItem': 101}
[PDFParser] Labels   : {'picture': 2, 'text': 59, 'section_header': 13, 'list_item': 101}
[PDFParser] Labels   : {'picture': 2, 'text': 59, 'section_header': 13, 'list_item': 101}
[PDFParser] Label samples:
  [text] (1): An official website of the United States government
  [text] (2): National Institutes of Health / National Library of Medicine
  [section_header] (1): Blood Glucose Test
  [section_header] (2): What is a blood glucose test?
  [list_item] (1): As part of a routine checkup [https://medlineplus.gov/healthcheckup.html]  to sc...
  [list_item] (2): To help diagnose the cause of symptoms that may mean your blood glucose is too h...
not describing image
not describing image

[PDFParser] Stored documents       : {'image': 2, 'text': 

[INFO] 2026-07-21 21:28:31,913 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-21 21:28:31,927 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\adity\OneDrive\Documents\MedMate Clinic Copilot\venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-07-21 21:28:31,931 [RapidOCR] main.py:63: Using C:\Users\adity\OneDrive\Documents\MedMate Clinic Copilot\venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-07-21 21:28:32,006 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-21 21:28:32,006 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\adity\OneDrive\Documents\MedMate Clinic Copilot\venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-21 21:28:32,006 [RapidOCR] main.py:63: Using C:\Users\adity\OneDrive\Documents\MedMate Clinic Copilot\venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-21 21:28:32


[PDFParser] File     : c-Reactive_protein_test.pdf
[PDFParser] Pages    : (1, 5)
[PDFParser] Images   : enabled
[PDFParser] Formulas : enabled (no VLM)
[PDFParser] Types    : {'PictureItem': 2, 'TextItem': 43, 'SectionHeaderItem': 13, 'ListItem': 33}
[PDFParser] Labels   : {'picture': 2, 'text': 43, 'section_header': 13, 'list_item': 33}
[PDFParser] Labels   : {'picture': 2, 'text': 43, 'section_header': 13, 'list_item': 33}
[PDFParser] Label samples:
  [text] (1): An official website of the United States government
  [text] (2): National Institutes of Health / National Library of Medicine
  [section_header] (1): C-Reactive Protein (CRP) Test
  [section_header] (2): What is a c-reactive (CRP) protein test?
  [list_item] (1): Infections from bacteria [https://medlineplus.gov/bacterialinfections.html] or v...
  [list_item] (2): Inflammatory bowel disease, disorders of the intestines that include Crohn's dis...
not describing image
not describing image

[PDFParser] Stored documents      

[INFO] 2026-07-21 21:28:40,362 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-21 21:28:40,362 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\adity\OneDrive\Documents\MedMate Clinic Copilot\venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-21 21:28:40,362 [RapidOCR] main.py:63: Using C:\Users\adity\OneDrive\Documents\MedMate Clinic Copilot\venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-21 21:28:40,438 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-21 21:28:40,454 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\adity\OneDrive\Documents\MedMate Clinic Copilot\venv\Lib\site-packages\rapidocr\models\PP-OCRv6_rec_small.onnx
[INFO] 2026-07-21 21:28:40,456 [RapidOCR] main.py:63: Using C:\Users\adity\OneDrive\Documents\MedMate Clinic Copilot\venv\Lib\site-packages\rapidocr\models\PP-OCRv6_rec_small.onnx
Loading weights: 100%|████

... (9 outputs left out)

[INFO] 2026-07-21 21:29:46,436 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-21 21:29:46,449 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\adity\OneDrive\Documents\MedMate Clinic Copilot\venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-07-21 21:29:46,449 [RapidOCR] main.py:63: Using C:\Users\adity\OneDrive\Documents\MedMate Clinic Copilot\venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-07-21 21:29:46,630 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-21 21:29:46,634 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\adity\OneDrive\Documents\MedMate Clinic Copilot\venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-21 21:29:46,634 [RapidOCR] main.py:63: Using C:\Users\adity\OneDrive\Documents\MedMate Clinic Copilot\venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-21 21:29:46


[PDFParser] File     : total_cholesterol_test.pdf
[PDFParser] Pages    : (1, 6)
[PDFParser] Images   : enabled
[PDFParser] Formulas : enabled (no VLM)
[PDFParser] Types    : {'PictureItem': 3, 'TextItem': 34, 'SectionHeaderItem': 12, 'ListItem': 38}
[PDFParser] Labels   : {'picture': 3, 'text': 34, 'section_header': 12, 'list_item': 38}
[PDFParser] Labels   : {'picture': 3, 'text': 34, 'section_header': 12, 'list_item': 38}
[PDFParser] Label samples:
  [text] (1): National Institutes of Health / National Library of Medicine
  [text] (2): Home → Medical Encyclopedia → Total cholesterol
  [section_header] (1): Total cholesterol
  [section_header] (2): How the Test is Performed
  [list_item] (1): Total cholesterol
  [list_item] (2): Low-density lipoprotein cholesterol (LDL cholesterol)
not describing image
not describing image
not describing image

[PDFParser] Stored documents       : {'image': 3, 'text': 35}
[PDFParser] Total documents         : 38
[PDFParser] Images on disk          : 

[INFO] 2026-07-21 21:29:58,158 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\adity\OneDrive\Documents\MedMate Clinic Copilot\venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-07-21 21:29:58,165 [RapidOCR] main.py:63: Using C:\Users\adity\OneDrive\Documents\MedMate Clinic Copilot\venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-07-21 21:29:58,345 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-21 21:29:58,351 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\adity\OneDrive\Documents\MedMate Clinic Copilot\venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-21 21:29:58,351 [RapidOCR] main.py:63: Using C:\Users\adity\OneDrive\Documents\MedMate Clinic Copilot\venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-21 21:29:58,428 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-21 21:29:58


[PDFParser] File     : triglycerides_test.pdf
[PDFParser] Pages    : (1, 7)
[PDFParser] Images   : enabled
[PDFParser] Formulas : enabled (no VLM)
[PDFParser] Types    : {'PictureItem': 2, 'TextItem': 59, 'SectionHeaderItem': 12, 'ListItem': 51}
[PDFParser] Labels   : {'picture': 2, 'text': 59, 'section_header': 12, 'list_item': 51}
[PDFParser] Labels   : {'picture': 2, 'text': 59, 'section_header': 12, 'list_item': 51}
[PDFParser] Label samples:
  [text] (1): National Institutes of Health / National Library of Medicine
  [text] (2): Home → Medical Tests → Triglycerides Test
  [section_header] (1): Triglycerides Test
  [section_header] (2): What is a triglycerides test?
  [list_item] (1): Eating a lot of foods that are high in fat [https://medlineplus.gov/dietaryfats....
  [list_item] (2): Not getting enough exercise [https://medlineplus.gov/healthrisksofaninactivelife...
not describing image
not describing image

[PDFParser] Stored documents       : {'image': 2, 'text': 60}
[PDFParse

In [43]:
document_metadata_levothyroxine = {
    "drug_name" : "levothyroxine",
    "document_type":"drug_info"
}

In [ ]:
parser = PDFParser(images_output_dir=RAG_IMAGES_PATH, process_images=True, document_metadata=document_metadata_lamotrigine)

[PDFParser] Formula processing DISABLED — formulas will be stored as plain text
[PDFParser] Pipeline options: process_images=True, process_formulas=False, do_formula_enrichment=False, describe_images=False


In [7]:
parsed_docs = parser.parse(r"C:\Users\adity\OneDrive\Documents\MedMate Clinic Copilot\data\rag_docs\Lamotrigine.pdf")
print("Total documents:", len(parsed_docs.documents))

[INFO] 2026-07-21 17:32:25,226 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-21 17:32:25,256 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\adity\OneDrive\Documents\MedMate Clinic Copilot\venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-07-21 17:32:25,259 [RapidOCR] main.py:63: Using C:\Users\adity\OneDrive\Documents\MedMate Clinic Copilot\venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-07-21 17:32:25,432 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-21 17:32:25,437 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\adity\OneDrive\Documents\MedMate Clinic Copilot\venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-21 17:32:25,438 [RapidOCR] main.py:63: Using C:\Users\adity\OneDrive\Documents\MedMate Clinic Copilot\venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-21 17:32:25


[PDFParser] File     : Lamotrigine.pdf
[PDFParser] Pages    : (33, 37)
[PDFParser] Images   : enabled
[PDFParser] Formulas : disabled
[PDFParser] Types    : {'TextItem': 40, 'SectionHeaderItem': 13, 'TableItem': 3}
[PDFParser] Labels   : {'text': 37, 'section_header': 13, 'table': 3, 'caption': 2, 'footnote': 1}
[PDFParser] Labels   : {'text': 37, 'section_header': 13, 'table': 3, 'caption': 2, 'footnote': 1}
[PDFParser] Label samples:
  [text] (1): mcg/mL is 4 to 6 times the trough plasma concentration observed in the controlle...
  [text] (2): Lamotrigine is metabolized predominantly by glucuronic acid conjugation; the maj...
  [section_header] (1): Metabolism
  [section_header] (2): Enzyme Induction
  [caption] (1): Table 7. Mean Pharmacokinetic Parameters of Immediate-Release Lamotrigine in Hea...
  [caption] (2): Table 8. Summary of Drug Interactions with Lamotrigine
  [footnote] (1): The majority of parameter means determined in each study had coefficients of var...
Page tables 

In [44]:
from pathlib import Path

pdf_path = r"C:\Users\adity\OneDrive\Documents\MedMate Clinic Copilot\data\rag_docs\levothyroxine.pdf"
total_pages = 5
batch_size = 10

all_documents = []
all_image_paths = []
failed_ranges = []

fig_offset, tbl_offset, formula_offset = 0, 0, 0

for batch_start in range(1, total_pages + 1, batch_size):
    batch_end = min(batch_start + batch_size - 1, total_pages)
    print(f"Processing pages {batch_start}-{batch_end}...")

    parser = PDFParser(
        images_output_dir=RAG_IMAGES_PATH,
        process_images=True,
        process_formulas=True,
        page_range=(batch_start, batch_end),
        figure_idx_offset=fig_offset,
        table_idx_offset=tbl_offset,
        formula_idx_offset=formula_offset,
        document_metadata=document_metadata_levothyroxine
    )

    try:
        parsed_doc = parser.parse(pdf_path)
        all_documents.extend(parsed_doc.documents)
        all_image_paths.extend(parsed_doc.image_paths)

        # bump global counters using the correct metadata key: "element_type"
        fig_offset += sum(1 for d in parsed_doc.documents if d.metadata.get("element_type") == "image")
        tbl_offset += sum(1 for d in parsed_doc.documents if d.metadata.get("element_type") == "table")
        formula_offset += sum(1 for d in parsed_doc.documents if d.metadata.get("element_type") == "formula")

        print(f"  -> OK, {len(parsed_doc.documents)} docs (running offsets: fig={fig_offset}, tbl={tbl_offset}, formula={formula_offset})")

    except Exception as e:
        print(f"  FAILED pages {batch_start}-{batch_end}: {type(e).__name__}: {e}")
        failed_ranges.append((batch_start, batch_end))

print("\n=== SUMMARY ===")
print("Total documents:", len(all_documents))
print("Total images:", len(all_image_paths))
print("Failed ranges:", failed_ranges)

[INFO] 2026-07-21 20:40:11,079 [RapidOCR] base.py:23: Using engine_name: onnxruntime


Processing pages 1-5...
[PDFParser] Formula detection ENABLED (fast, layout-only — no VLM)
[PDFParser] Pipeline options: process_images=True, process_formulas=True, do_formula_enrichment=False, describe_images=False


[INFO] 2026-07-21 20:40:11,093 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\adity\OneDrive\Documents\MedMate Clinic Copilot\venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-07-21 20:40:11,093 [RapidOCR] main.py:63: Using C:\Users\adity\OneDrive\Documents\MedMate Clinic Copilot\venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-07-21 20:40:11,210 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-21 20:40:11,217 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\adity\OneDrive\Documents\MedMate Clinic Copilot\venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-21 20:40:11,217 [RapidOCR] main.py:63: Using C:\Users\adity\OneDrive\Documents\MedMate Clinic Copilot\venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-21 20:40:11,281 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-21 20:40:11


[PDFParser] File     : levothyroxine.pdf
[PDFParser] Pages    : (1, 5)
[PDFParser] Images   : enabled
[PDFParser] Formulas : enabled (no VLM)
[PDFParser] Types    : {'SectionHeaderItem': 23, 'TextItem': 86, 'ListItem': 1, 'PictureItem': 2, 'TableItem': 2}
[PDFParser] Labels   : {'section_header': 23, 'text': 86, 'list_item': 1, 'picture': 2, 'table': 2}
[PDFParser] Labels   : {'section_header': 23, 'text': 86, 'list_item': 1, 'picture': 2, 'table': 2}
[PDFParser] Label samples:
  [section_header] (1): LEVOTHYROXINE POWDER- levothyroxine sodium powder Covetrus North America
  [section_header] (2): Levothyroxine Powder
  [text] (1): Disclaimer: This drug has not been found by FDA to be safe and effective, and th...
  [text] (2): Indications: For use in horses for correction of conditions associated with low ...
  [list_item] (1): ----------
not describing image
not describing image

[PDFParser] Stored documents       : {'text': 85, 'image': 2, 'table': 2}
[PDFParser] Total documents    

In [8]:
for idx, doc in enumerate(all_documents):

    print("=" * 100)

    print(f"Element #{idx}")

    print("page: ", doc.metadata.get("page"))

    print("Element type: ", doc.metadata.get("element_type"))

    print("Metadata: ", doc.metadata)

    print("Section:",
          doc.metadata.get("section"))

    print("\nContent:\n")

    print(doc.page_content)

Element #0
page:  1
Element type:  image
Metadata:  {'source': 'blood_glucose_test.pdf', 'source_path': 'C:\\Users\\adity\\OneDrive\\Documents\\MedMate Clinic Copilot\\data\\rag_docs\\lab_glossary\\blood_glucose_test.pdf', 'doc_type': 'pdf', 'parser': 'docling', 'parser_version': '2.0', 'section': 'Document', 'parent_section': None, 'header_path': ['Document'], 'header_path_str': 'Document', 'page': 1, 'test_name': 'Blood Glucose', 'test_slug': 'blood_glucose', 'document_source': 'MedlinePlus', 'document_type': 'lab_glossary', 'element_type': 'image', 'chunk_strategy': 'preserve', 'image_index': 1, 'image_path': 'C:\\Users\\adity\\OneDrive\\Documents\\MedMate Clinic Copilot\\data\\rag_storage\\images\\blood_glucose_test-figure-1.png', 'has_image': True, 'caption': '', 'image_width': 36, 'image_height': 25}
Section: Document

Content:

[Image]
Element #1
page:  1
Element type:  text
Metadata:  {'source': 'blood_glucose_test.pdf', 'source_path': 'C:\\Users\\adity\\OneDrive\\Documents\\Me

In [9]:
import src.medmate.rag.chunker as pdf_chunker

chunker = pdf_chunker.PDFChunker(
    chunk_size=800,
    chunk_overlap=150,
)

In [10]:
from src.medmate.rag.helpers.parsed_document import ParsedDocument

parsed_doc_for_test = ParsedDocument(
    documents=all_documents,
    image_paths=all_image_paths,
    source_type="pdf",
)

In [11]:
chunks = chunker.chunk(parsed_doc_for_test)

print("Total chunks:", len(chunks))

Total chunks: 254


In [12]:
for i, chunk in enumerate(chunks):

    print("=" * 80)
    
    print("Chunk:", i)

    print("\nMetadata:")
    print(chunk.metadata)

    print("\nContent:")
    print(chunk.page_content)

    print("\n")

Chunk: 0

Metadata:
{'source': 'blood_glucose_test.pdf', 'source_path': 'C:\\Users\\adity\\OneDrive\\Documents\\MedMate Clinic Copilot\\data\\rag_docs\\lab_glossary\\blood_glucose_test.pdf', 'doc_type': 'pdf', 'parser': 'docling', 'parser_version': '2.0', 'section': 'Document', 'parent_section': None, 'header_path': ['Document'], 'header_path_str': 'Document', 'page': 1, 'test_name': 'Blood Glucose', 'test_slug': 'blood_glucose', 'document_source': 'MedlinePlus', 'document_type': 'lab_glossary', 'element_type': 'text', 'chunk_strategy': 'split', 'chunk_index': 0, 'chunk_total': 1, 'page_start': 1, 'page_end': 1, 'group_id': 'ee17c163d77b', 'chunk_id': 'fc2fdaa546f11889f0906fa8c90e3ae5'}

Content:
An official website of the United States government

National Institutes of Health / National Library of Medicine

Home → Medical Tests → Blood Glucose Test

URL of this page: https://medlineplus.gov/lab-tests/blood-glucose-test/


Chunk: 1

Metadata:
{'source': 'blood_glucose_test.pdf', 'sour

In [3]:
from src.medmate.rag.embeddings import EmbeddingService



c:\Users\adity\OneDrive\Documents\MedMate Clinic Copilot\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
texts = [
    chunk.page_content
    for chunk in chunks
]

In [4]:
embedding_service = EmbeddingService()

In [15]:
import time
start = time.time()

vectors = embedding_service.embed_docs(texts)
print("Time:", time.time() - start)
print("Vectors:", len(vectors))
print("Dimensions:", len(vectors[0]))

Time: 3.407097578048706
Vectors: 254
Dimensions: 1536


In [5]:
from src.medmate.rag.vector_store import QdrantService

qdrant = QdrantService(
    collection_name="medmate_rag_docs"
)

print("Qdrant initialized")

Qdrant initialized


In [13]:
qdrant.client.close()

In [20]:
try:
    qdrant.delete_collection()
except Exception:
    pass

qdrant.create_collection(
    vector_size=1536
)

print("Collection created")

Collection created


c:\Users\adity\OneDrive\Documents\MedMate Clinic Copilot\src\medmate\rag\vector_store.py:123: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  self.client.create_payload_index(


In [17]:
qdrant.upsert_documents(
    chunks=chunks,
    dense_vectors=vectors,
)

254

In [12]:
info = qdrant.client.get_collection("medmate_rag_docs")

print(info.points_count)

1422


In [6]:
from src.medmate.rag.hybrid_search import HybridRetriever
from src.medmate.rag.reranker import CohereRanker

cohere_reranker = CohereRanker()


In [7]:
retriever = HybridRetriever(qdrant, cohere_reranker)

In [ ]:
query = "How does ethinylestradiol affect the clearance of lamotrigine?"

In [9]:
query_dense = embedding_service.embed_query(query)

In [61]:
dense_results = retriever.search_dense_only(
    query_dense_vector=query_dense,
    limit=7,
)

In [62]:
print("\n===== DENSE SEARCH =====\n")

for idx, result in enumerate(dense_results, start=1):
    print(f"Rank {idx}")
    print(result)
    print(f"Score: {result.score:.4f}")
    print(result.document.page_content)
    
    print("-" * 80)


===== DENSE SEARCH =====

Rank 1
SearchResult(score=0.6473, type=dense, source=metformin.pdf, page=10, element=text, section=Alogliptin and Metformin HCl, preview='In a pooled analysis of these four controlled clinical studies, the overall inci')
Score: 0.6473
In a pooled analysis of these four controlled clinical studies, the overall incidence of adverse reactions was 74% in patients treated with alogliptin and metformin HCl tablets compared to 75% treated with placebo. Overall discontinuation of therapy due to adverse reactions was 6.2% with alogliptin and metformin HCl tablets compared to 1.9% in placebo, 6.4% in metformin and 5.0% in alogliptin.

Adverse reactions reported in ≥4% of patients treated with alogliptin and metformin HCl tablets and more frequently than in patients who received alogliptin, metformin or placebo are summarized in Table 1.
--------------------------------------------------------------------------------
Rank 2
SearchResult(score=0.6406, type=dense, source=

In [30]:
hybrid_results = retriever.search(
    query=query,
    query_dense_vector=query_dense,
    limit=10,
)

In [31]:
print("\n===== HYBRID SEARCH =====\n")

for idx, result in enumerate(hybrid_results, start=1):
    print(f"Rank {idx}")
    print(result)
    print(f"Score: {result.score:.4f}")
    print(result.document.page_content)
    print("-" * 80)


===== HYBRID SEARCH =====

Rank 1
SearchResult(score=0.5000, type=hybrid, source=Lamotrigine.pdf, page=45, element=text, section=Suicidal Thinking and Behavior, preview='Inform patients, their caregivers, and families that AEDs, including lamotrigine')
Score: 0.5000
Inform patients, their caregivers, and families that AEDs, including lamotrigine extendedrelease tablets, may increase the risk of suicidal thoughts and behavior. Instruct them to be alert for the emergence or worsening of symptoms of depression, any unusual changes in mood or behavior, or the emergence of suicidal thoughts or behavior or thoughts about self-harm. Instruct them to immediately report behaviors of concern to their healthcare providers.
--------------------------------------------------------------------------------
Rank 2
SearchResult(score=0.5000, type=hybrid, source=Lamotrigine.pdf, page=49, element=text, section=Call your healthcare provider right away if you have any of the following symptoms:, preview='

In [10]:
results = retriever.retrieve(query=query, query_dense_vector=query_dense)

def print_results(context) -> None:
    print(f"\n{'='*60}")
    print(f"Total chunks retrieved: {len(context.chunks)}")
    print(f"Total context chars: {context.total_chars}")
    print(f"{'='*60}\n")

    for i, doc in enumerate(context.chunks):
        meta = doc.metadata
        print(f"Chunk {i+1}/{len(context.chunks)}")
        print(f"  Section   : {meta.get('header_path_str', '?')}")
        print(f"  Type      : {meta.get('element_type', '?')}")
        print(f"  Page      : {meta.get('page_start', '?')} - {meta.get('page_end', '?')}")
        print(f"  Chunk     : {meta.get('chunk_index', '?')} of {meta.get('chunk_total', '?')}")
        print(f"  Group ID  : {meta.get('group_id', '?')}")
        print(f"  Source    : {meta.get('source', '?')}")
        print(f"  Content   :")
        print(f"{'-'*60}")
        print(doc.page_content)
        print(f"{'='*60}\n")

print_results(results)

Called cohere reranker

Total chunks retrieved: 5
Total context chars: 2995

Chunk 1/5
  Section   : Why do I need a potassium blood test?
  Type      : text
  Page      : 2 - 2
  Chunk     : 0 of 2
  Group ID  : dccace0e0943
  Source    : potassium_test.pdf
  Content   :
------------------------------------------------------------
Your health care provider may order a potassium blood test as part of your routine checkup [https://medlineplus.gov/healthcheckup.html] or to monitor an existing condition, such as diabetes [https://medlineplus.gov/diabetes.html] , kidney disease, or adrenal gland disorders [https://medlineplus.gov/adrenalglanddisorders.html] . You may also need this test if you take medicines that could affect your potassium levels or if you have symptoms of having too much or too little potassium.

Chunk 2/5
  Section   : What do the results mean?
  Type      : text
  Page      : 4 - 5
  Chunk     : 1 of 3
  Group ID  : 7dfabb46aa51
  Source    : triglycerides_test.pdf
  C

In [11]:
from src.medmate.rag.helpers.generation.schemas import RetrievedDocument
documents = []

for chunk in results.chunks:

    metadata = chunk.metadata

    documents.append(
        RetrievedDocument(
            content=chunk.page_content,
            source=metadata.get("source"),
            section=metadata.get("header_path_str"),
            page_start=metadata.get("page_start"),
            page_end=metadata.get("page_end"),
            element_type=metadata.get("element_type"),
        )
    )

In [12]:
from src.medmate.rag.generator import RAGGenerator
generator = RAGGenerator()

response = generator.generate(
    question=query,
    documents=documents,
)

print("\nANSWER")
print("=" * 50)
print(response.answer)

print("\nSOURCES")
print("=" * 50)

for source in response.sources:
    print(source.model_dump())


ANSWER
To lower blood triglycerides, lifestyle changes include eating heart-healthy foods while limiting added sugar and foods high in saturated fat, getting regular physical activity, aiming for a healthy weight, managing stress, quitting smoking, getting enough sleep, and limiting alcohol intake. A healthcare provider may order a potassium blood test during a routine checkup to monitor or diagnose conditions related to abnormal potassium levels, such as kidney disease, high blood pressure, heart disease, diabetes, or adrenal gland disorders, or if you take medicines that could affect potassium levels or have symptoms of abnormal potassium.

SOURCES
{'source': 'potassium_test.pdf', 'section': 'Why do I need a potassium blood test?', 'page_start': 2, 'page_end': 2}
{'source': 'triglycerides_test.pdf', 'section': 'What do the results mean?', 'page_start': 4, 'page_end': 5}
{'source': 'potassium_test.pdf', 'section': 'What is it used for?', 'page_start': 1, 'page_end': 1}
